# 02 — Chọn đặc trưng và mô hình cây quyết định

**Mục tiêu:** so sánh mô hình CART dùng toàn bộ đặc trưng với lựa chọn đặc trưng Filter và Wrapper; báo cáo Recall, ROC-AUC và trích luật IF–THEN. Notebook được tạo ở trạng thái chưa chạy.

> Đây là bài toán phân loại theo nhãn hiện có trong bộ dữ liệu, không thay thế chẩn đoán y khoa. Xác nhận ý nghĩa cột nhãn trước khi diễn giải kết quả.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay,
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import find_target_column, find_identifier_columns

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_PATH = DATA_DIR / "parkinsons_processed.csv"
RAW_PATH = DATA_DIR / "D07 - parkinsons.data.xlsx"
RANDOM_STATE = 42
TEST_SIZE = 0.20


## 1. Đọc dữ liệu và xác định nhãn

Ưu tiên CSV làm việc nếu notebook 01 đã tạo file; nếu không, đọc file gốc. Sửa `TARGET_COL` thủ công khi tên nhãn của dataset khác `status`/`target`/`label`.


In [ ]:
if PROCESSED_PATH.exists():
    df = pd.read_csv(PROCESSED_PATH)
elif RAW_PATH.exists():
    df = pd.read_excel(RAW_PATH)
else:
    raise FileNotFoundError(
        f"Không tìm thấy {PROCESSED_PATH} hoặc {RAW_PATH}. "
        "Đặt dữ liệu vào thư mục data/ trước khi chạy."
    )

df.columns = [str(c).strip() for c in df.columns]
TARGET_COL = find_target_column(df.columns)
# Nếu không tự nhận diện được, thay None bằng đúng tên cột mục tiêu, ví dụ: TARGET_COL = "status"
if TARGET_COL is None:
    raise ValueError("Không nhận diện được cột nhãn. Hãy gán TARGET_COL bằng đúng tên cột mục tiêu.")

ID_COLS = find_identifier_columns(df.columns)
df = df.drop_duplicates().dropna(subset=[TARGET_COL]).copy()
print("Target:", TARGET_COL, "| ID columns:", ID_COLS)
display(df.head())
display(df[TARGET_COL].value_counts(dropna=False).to_frame("count"))


## 2. Chuẩn bị X, y và tách train/test

Các cột định danh bị loại. Mô hình cây sử dụng đặc trưng số; những cột phi số khác sẽ không được dùng ở phiên bản baseline này. Tập test được giữ lại để đánh giá cuối cùng.


In [ ]:
drop_cols = [TARGET_COL, *ID_COLS]
X_all = df.drop(columns=drop_cols, errors="ignore").select_dtypes(include=np.number).copy()
y_raw = df[TARGET_COL].copy()

if X_all.shape[1] == 0:
    raise ValueError("Không có đặc trưng số sau khi bỏ cột nhãn/định danh. Kiểm tra dữ liệu đầu vào.")
if y_raw.nunique(dropna=True) < 2:
    raise ValueError("Cột mục tiêu cần có ít nhất 2 lớp để phân loại.")

# Với nhãn Parkinsons phổ biến, status là 0/1. Nếu nhãn khác, ánh xạ về mã số để đánh giá.
if set(pd.Series(y_raw.dropna().unique()).tolist()).issubset({0, 1}):
    y = y_raw.astype(int)
else:
    y, class_labels = pd.factorize(y_raw, sort=True)
    y = pd.Series(y, index=y_raw.index, name=TARGET_COL)
    print("Mã hóa nhãn theo thứ tự:", dict(enumerate(class_labels)))

# Điền thiếu bằng median đặc trưng; median học trên train sẽ tốt hơn về chống leakage.
# Để đơn giản và nhất quán, loại cột toàn thiếu trước rồi học median trên X_train bên dưới.
X_all = X_all.loc[:, X_all.notna().any(axis=0)]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)
feature_cols = X_train.columns.tolist()
print(f"Train: {X_train.shape}; Test: {X_test.shape}; Số đặc trưng: {len(feature_cols)}")


## 3. Mô hình cơ sở: CART với toàn bộ đặc trưng

`DecisionTreeClassifier(criterion='gini')` là cây quyết định CART. Có thể thử `criterion='entropy'` như một lựa chọn dựa trên entropy, nhưng không nên đồng nhất trực tiếp với bản cài đặt nguyên bản của ID3/C4.5.


In [ ]:
cart_all = DecisionTreeClassifier(
    criterion="gini",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    min_samples_leaf=2,
)
cart_all.fit(X_train, y_train)


## 4. Feature Selection — Filter

Dùng `mutual_info_classif` để xếp hạng mức độ liên quan giữa từng đặc trưng và nhãn. Bộ chọn chỉ fit trên tập train để tránh dùng thông tin từ test.


In [ ]:
k = min(10, X_train.shape[1])
filter_selector = SelectKBest(score_func=mutual_info_classif, k=k)
filter_selector.fit(X_train, y_train)
filter_mask = filter_selector.get_support()
filter_features = X_train.columns[filter_mask].tolist()
filter_scores = pd.DataFrame({
    "feature": X_train.columns,
    "score": filter_selector.scores_,
}).sort_values("score", ascending=False, na_position="last")
print(f"Số đặc trưng được chọn: {len(filter_features)}")
display(filter_scores)

cart_filter = DecisionTreeClassifier(
    criterion="gini", class_weight="balanced",
    random_state=RANDOM_STATE, min_samples_leaf=2,
)
cart_filter.fit(X_train[filter_features], y_train)


## 5. Feature Selection — Wrapper (RFECV)

RFECV lặp lại việc loại đặc trưng trong cross-validation trên tập train. Ví dụ dưới đây chọn theo Recall; với dữ liệu y tế, vẫn cần xem xét đánh đổi với Precision và hiệu năng tổng thể.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
wrapper_selector = RFECV(
    estimator=DecisionTreeClassifier(
        criterion="gini", class_weight="balanced", random_state=RANDOM_STATE, min_samples_leaf=2
    ),
    step=1,
    cv=cv,
    scoring="recall",
    min_features_to_select=min(5, X_train.shape[1]),
    n_jobs=-1,
)
wrapper_selector.fit(X_train, y_train)
wrapper_features = X_train.columns[wrapper_selector.support_].tolist()
print(f"RFECV chọn {len(wrapper_features)} đặc trưng:", wrapper_features)
print("Estimator cuối cùng bên trong RFECV đã fit trên các đặc trưng được chọn.")


## 6. Hàm đánh giá và so sánh mô hình

Báo cáo Accuracy, Precision, Recall, F1, ROC-AUC và confusion matrix. Với nhãn 0/1, mặc định lớp dương là `1`; xác nhận mã nhãn trước khi diễn giải Recall như khả năng bắt đúng lớp Parkinson.


In [ ]:
POS_LABEL = 1

def evaluate_model(name, model, X_eval, y_true):
    y_pred = model.predict(X_eval)
    y_score = None
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_eval)
        classes = list(model.classes_)
        if POS_LABEL in classes:
            y_score = probabilities[:, classes.index(POS_LABEL)]
        elif probabilities.shape[1] == 2:
            y_score = probabilities[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_eval)

    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "roc_auc": np.nan,
    }
    if y_score is not None and pd.Series(y_true).nunique() == 2:
        try:
            row["roc_auc"] = roc_auc_score(y_true, y_score)
        except ValueError:
            pass

    print(f"\n===== {name} =====")
    print(classification_report(y_true, y_pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, values_format="d")
    plt.title(f"Confusion matrix — {name}")
    plt.tight_layout()
    plt.show()
    return row, y_score

results = []
roc_data = []

row, score = evaluate_model("CART - all features", cart_all, X_test[feature_cols], y_test)
results.append(row)
if score is not None: roc_data.append(("CART - all features", score))

row, score = evaluate_model("Filter + CART", cart_filter, X_test[filter_features], y_test)
results.append(row)
if score is not None: roc_data.append(("Filter + CART", score))

row, score = evaluate_model("Wrapper RFECV", wrapper_selector, X_test[feature_cols], y_test)
results.append(row)
if score is not None: roc_data.append(("Wrapper RFECV", score))

metrics_df = pd.DataFrame(results).sort_values("recall", ascending=False)
display(metrics_df.round(4))


## 7. Đường cong ROC và ROC-AUC


In [ ]:
if roc_data and pd.Series(y_test).nunique() == 2:
    plt.figure(figsize=(8, 6))
    for name, scores in roc_data:
        RocCurveDisplay.from_predictions(y_test, scores, name=name, ax=plt.gca())
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
    plt.title("So sánh ROC curves")
    plt.tight_layout()
    plt.show()
else:
    print("Không thể vẽ ROC: cần bài toán hai lớp và điểm số xác suất/decision score.")


## 8. Trực quan hóa cây và trích luật IF–THEN

Cây được giới hạn độ sâu khi **vẽ** để dễ đọc; mô hình đánh giá phía trên vẫn dùng cây đã fit với cấu hình đã khai báo. Luật xuất ra là luật kỹ thuật từ cây sklearn, cần kiểm tra tính hợp lý và tránh diễn giải thành quy tắc chẩn đoán y khoa.


In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    cart_all,
    feature_names=feature_cols,
    class_names=[str(c) for c in cart_all.classes_],
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=8,
)
plt.title("Cây quyết định CART — hiển thị tối đa 4 tầng")
plt.tight_layout()
plt.show()

print("Luật IF–THEN của CART (toàn bộ đặc trưng):")
print(export_text(cart_all, feature_names=feature_cols, decimals=3))

print("Luật IF–THEN của Filter + CART:")
print(export_text(cart_filter, feature_names=filter_features, decimals=3))

print("Luật IF–THEN của Wrapper (các đặc trưng được chọn):")
wrapper_tree = wrapper_selector.estimator_
print(export_text(wrapper_tree, feature_names=wrapper_features, decimals=3))


## 9. Kết luận mô hình — điền sau khi chạy

- Mô hình có Recall cao nhất:
- Precision và ROC-AUC tương ứng:
- Số đặc trưng sau Filter / Wrapper:
- Các đặc trưng được chọn có hợp lý về mặt dữ liệu không?
- Luật IF–THEN nổi bật và ý nghĩa trong phạm vi bộ dữ liệu:
- Hạn chế: kích thước/mức đại diện của mẫu, mất cân bằng lớp, khả năng tổng quát hóa, không dùng cho chẩn đoán lâm sàng:
